# Ablation experiments

Research notebook for comparing complete and stage-disabled workflow variants. It writes generated outputs and metric artifacts under `table2text_pydanticai/evaluation/`.


## Canonical stage-ablation study


In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    score_reference_metrics_for_notebook,
    score_deepeval_for_notebook,
)
from table2text.evaluation.datasets import read_examples, write_jsonl

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

DATASET_ID = "sportsett_basketball"
EXAMPLE_ID = "4934"

RUN_REFERENCE_METRICS = True
RUN_DEEPEVAL = False  # Run after the first smoke pass; this is slower.

experiment_name = (
    f"ablation_sportsett_{EXAMPLE_ID}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

existing_generations_path = (
    project_dir
    / "evaluation/generations/"
    / "five_dataset_five_each_comparison_20260804_205019_sportsett_basketball_combined_generations.jsonl"
)

ablation_variant_ids = [
    "no_insight_synthesis",
    "no_writer_quality_revision",
    "no_audit_repair_rounds",
]

def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)

def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def write_generation_records(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

def compact_metric_table(scores):
    if scores.empty:
        return pd.DataFrame()

    return (
        scores[scores["status"].isin(["scored", "error", "skipped", "unavailable"])]
        .pivot_table(
            index=["dataset_id", "example_id", "metric_name"],
            columns="variant_id",
            values="score",
            aggfunc="first",
        )
        .reset_index()
    )

def delta_vs_full(scores):
    table = compact_metric_table(scores)
    if table.empty or "full_system" not in table.columns:
        return table

    lower_is_better = {"ter", "corpus_ter"}
    variant_cols = [
        c for c in table.columns
        if c not in {"dataset_id", "example_id", "metric_name", "full_system"}
    ]

    rows = []
    for _, row in table.iterrows():
        metric_name = row["metric_name"]
        full_score = row.get("full_system")
        if pd.isna(full_score):
            continue

        for variant_id in variant_cols:
            variant_score = row.get(variant_id)
            if pd.isna(variant_score):
                continue

            if metric_name in lower_is_better:
                advantage_over_full = full_score - variant_score
            else:
                advantage_over_full = variant_score - full_score

            rows.append(
                {
                    "dataset_id": row["dataset_id"],
                    "example_id": row["example_id"],
                    "metric_name": metric_name,
                    "variant_id": variant_id,
                    "variant_score": variant_score,
                    "full_system_score": full_score,
                    "advantage_over_full": advantage_over_full,
                }
            )

    return pd.DataFrame(rows)

def summarise_generations(rows):
    frame = pd.DataFrame(rows)
    cols = [
        "dataset_id",
        "example_id",
        "variant_id",
        "error",
        "release_status",
        "writer_mode",
        "repair_rounds_used",
        "audit_support_rate",
        "elapsed_seconds",
    ]
    return frame[[c for c in cols if c in frame.columns]]

# Select the fixed example.
examples = read_examples(paths["prepared_examples"])
example = next(
    e for e in examples
    if e.dataset_id == DATASET_ID and str(e.example_id) == EXAMPLE_ID
)

examples_path = project_dir / f"evaluation/prepared/{experiment_name}.jsonl"
write_jsonl(examples_path, [example])

log(f"Selected {DATASET_ID}/{EXAMPLE_ID}")
log(f"Reference count: {len(example.references)}")

# Reuse existing raw baseline and full-system records.
existing_rows = read_jsonl(existing_generations_path)
baseline_rows = [
    row for row in existing_rows
    if row["dataset_id"] == DATASET_ID
    and str(row["example_id"]) == EXAMPLE_ID
    and row["variant_id"] in {"raw_deepseek_v4_flash", "full_system"}
]

found_baselines = sorted(row["variant_id"] for row in baseline_rows)
log(f"Loaded existing baseline rows: {found_baselines}")

if set(found_baselines) != {"raw_deepseek_v4_flash", "full_system"}:
    raise RuntimeError(f"Missing baseline rows. Found: {found_baselines}")

# Build ablation-only variants.
variants_payload = json.loads(
    (project_dir / "evaluation/config/variants_ablation.json").read_text(encoding="utf-8")
)
ablation_variants = [
    {**variant, "enabled": variant["variant_id"] in ablation_variant_ids}
    for variant in variants_payload["variants"]
    if variant["variant_id"] in ablation_variant_ids
]

variants_path = project_dir / f"evaluation/config/variants_{experiment_name}.json"
write_json(variants_path, {"variants": ablation_variants})

log(f"Ablation variants: {[variant['variant_id'] for variant in ablation_variants]}")

# Generate the missing ablations.
ablation_generations_path = (
    project_dir / f"evaluation/generations/{experiment_name}_ablation_only_generations.jsonl"
)
run_root = project_dir / f"evaluation/generations/{experiment_name}_runs"

start = time.perf_counter()
ablation_frame = await generate_reports_for_notebook(
    project_dir,
    examples_path=examples_path,
    variants_path=variants_path,
    output_path=ablation_generations_path,
    run_root=run_root,
    resume=False,
)
log(f"Finished ablation generation in {time.perf_counter() - start:.1f}s")

display(
    ablation_frame[
        [
            "dataset_id",
            "example_id",
            "variant_id",
            "error",
            "release_status",
            "writer_mode",
            "repair_rounds_used",
            "audit_support_rate",
            "elapsed_seconds",
        ]
    ]
)

# Combine existing raw/full with newly generated ablations.
combined_rows = baseline_rows + ablation_frame.to_dict("records")
combined_generations_path = (
    project_dir / f"evaluation/generations/{experiment_name}_combined_generations.jsonl"
)
write_generation_records(combined_generations_path, combined_rows)

log(f"Combined generations: {combined_generations_path}")

print("\nGENERATION SUMMARY")
display(summarise_generations(combined_rows))

print("\nOUTPUT PREVIEWS")
for row in combined_rows:
    print("\n" + "=" * 100)
    print(row["variant_id"])
    print("-" * 100)
    print((row.get("generated_text") or "")[:2500])

if RUN_REFERENCE_METRICS:
    reference_scores_path = (
        project_dir / f"evaluation/results/{experiment_name}_reference_metrics.jsonl"
    )
    metrics_path = project_dir / "evaluation/config/archive/metrics_ablation_sportsett_4934.json"

    log("Scoring compact reference metrics")
    reference_scores = score_reference_metrics_for_notebook(
        project_dir,
        generations_path=combined_generations_path,
        metric_config_path=metrics_path,
        output_path=reference_scores_path,
        include_ineligible=True,
    )

    print("\nREFERENCE METRICS")
    display(compact_metric_table(reference_scores))

    print("\nDELTA VS FULL SYSTEM")
    display(
        delta_vs_full(reference_scores)
        .sort_values(["metric_name", "advantage_over_full"])
        .reset_index(drop=True)
    )

if RUN_DEEPEVAL:
    deepeval_scores_path = (
        project_dir / f"evaluation/results/{experiment_name}_deepeval_metrics.jsonl"
    )
    metrics_path = project_dir / "evaluation/config/archive/metrics_ablation_sportsett_4934.json"

    log("Scoring DeepEval metrics")
    deepeval_scores = score_deepeval_for_notebook(
        project_dir,
        generations_path=combined_generations_path,
        metric_config_path=metrics_path,
        output_path=deepeval_scores_path,
        resume=False,
    )

    print("\nDEEPEVAL METRICS")
    display(compact_metric_table(deepeval_scores))

    print("\nDEEPEVAL DELTA VS FULL SYSTEM")
    display(
        delta_vs_full(deepeval_scores)
        .sort_values(["metric_name", "advantage_over_full"])
        .reset_index(drop=True)
    )

## Generic-request and model comparison experiment


In [ ]:
import json
import os
import time
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI

from table2text import Settings, Table2TextWorkflow
from table2text.config import load_env_files
from table2text.evaluation import default_paths, score_reference_metrics_for_notebook
from table2text.evaluation.datasets import read_examples, write_jsonl
from table2text.evaluation.models import GenerationBackend, GenerationRecord
from table2text.schemas import AuditMode, EvaluationFieldPolicy

project_dir = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
paths = default_paths(project_dir)

load_env_files([
    project_dir.parent / ".env",
    project_dir / ".env",
    project_dir.parent / ".env.local",
    project_dir / ".env.local",
])

dataset_id = "sportsett_basketball"
example_id = "4934"
GENERIC_REQUEST = "Understand the supplied data and report its strongest supported findings."

RAW_MODEL = os.getenv("T2T_RAW_BASELINE_MODEL", "deepseek-v4-flash")
RAW_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
RAW_MAX_SOURCE_CHARS = int(os.getenv("T2T_RAW_BASELINE_MAX_SOURCE_CHARACTERS", "150000"))
RAW_MAX_OUTPUT_TOKENS = int(os.getenv("T2T_RAW_BASELINE_MAX_OUTPUT_TOKENS", "3000"))

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"generic_only_{dataset_id}_{example_id}_{run_tag}"

def show(title, text):
    display(Markdown(f"## {title}\n\n{text.strip() if text and text.strip() else '*No text generated.*'}"))

def write_generation_records(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(record.model_dump_json() + "\n")

def metric_table(scores):
    if scores.empty:
        return pd.DataFrame()
    return (
        scores.pivot_table(
            index=["dataset_id", "example_id", "metric_name"],
            columns="variant_id",
            values="score",
            aggfunc="first",
        )
        .reset_index()
    )

examples = read_examples(paths["prepared_examples"])
example = next(
    e for e in examples
    if e.dataset_id == dataset_id and str(e.example_id) == str(example_id)
)

source_text = json.dumps(example.source_payload, indent=2, ensure_ascii=False)

run_root = project_dir / f"evaluation/generations/{experiment_name}_runs"
input_path = run_root / "_inputs" / f"{dataset_id}_{example_id}_source_only.json"
input_path.parent.mkdir(parents=True, exist_ok=True)
input_path.write_text(source_text + "\n", encoding="utf-8")

records = []

# =========================
# full_generic
# =========================

print("Running full_generic...")
t0 = time.perf_counter()

settings = replace(
    Settings.from_env(),
    output_dir=run_root / "full_generic" / dataset_id,
    random_seed=42,
)

workflow = Table2TextWorkflow(settings)

result = await workflow.run(
    inputs=[input_path],
    request=GENERIC_REQUEST,
    audit_mode=AuditMode.INTERNAL,
    evaluation_field_policy=EvaluationFieldPolicy(
        operational_input_paths=["$"],
        held_out_reference_paths=[],
        metadata_paths=[],
    ),
    report_genre=None,
    communication_task=None,
    output_form=None,
    focus_scope=None,
)

full_elapsed = time.perf_counter() - t0
full_output = result.final_writer_output
pipeline_result_path = settings.output_dir / result.run_id / "pipeline_result.json"
pipeline_result_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")

full_record = GenerationRecord(
    generation_id=f"{dataset_id}__{example_id}__full_generic__r0__s42",
    dataset_id=dataset_id,
    example_id=str(example_id),
    variant_id="full_generic",
    repetition=0,
    seed=42,
    task_family=example.task_family,
    output_mode=example.output_mode,
    language=example.language,
    source_text=source_text,
    references=example.references,
    parent_table=example.parent_table,
    request=GENERIC_REQUEST,
    generated_text=full_output.markdown,
    backend=GenerationBackend.TABLE2TEXT,
    run_id=result.run_id,
    pipeline_result_path=pipeline_result_path,
    writer_mode=full_output.writer_mode,
    release_status=result.release_status.value,
    approved_for_release=result.approved_for_release,
    primary_evaluation_eligible=result.primary_evaluation_eligible,
    primary_evaluation_reason=result.primary_evaluation_reason,
    elapsed_seconds=full_elapsed,
    metadata={
        "generic_contract_test": True,
        "explicit_contract_supplied": False,
    },
)

records.append(full_record)
print(f"full_generic done in {full_elapsed:.1f}s")
show("full_generic", full_record.generated_text)

# =========================
# raw_generic
# =========================

print("Running raw_generic...")
t0 = time.perf_counter()

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise RuntimeError("DEEPSEEK_API_KEY is missing from .env")

raw_source = source_text[:RAW_MAX_SOURCE_CHARS]
if len(source_text) > RAW_MAX_SOURCE_CHARS:
    raw_source += "\n\n[Source truncated.]"

raw_model = RAW_MODEL.replace("deepseek:", "")
client = OpenAI(api_key=api_key, base_url=RAW_BASE_URL)

messages = [
    {
        "role": "system",
        "content": (
            "You are a raw single-LLM data-to-text baseline. Use only the supplied "
            "source data and the user request. Do not use outside knowledge. "
            "Do not invent facts, numbers, entities, chronology, or causal explanations. "
            "Write the final answer only."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Request:\n{GENERIC_REQUEST}\n\n"
            f"Source data:\n{raw_source}"
        ),
    },
]

response = client.chat.completions.create(
    model=raw_model,
    messages=messages,
    temperature=0.2,
    max_tokens=RAW_MAX_OUTPUT_TOKENS,
)

raw_text = (response.choices[0].message.content or "").strip()
raw_elapsed = time.perf_counter() - t0

usage = getattr(response, "usage", None)
raw_record = GenerationRecord(
    generation_id=f"{dataset_id}__{example_id}__raw_generic__r0__s42",
    dataset_id=dataset_id,
    example_id=str(example_id),
    variant_id="raw_generic",
    repetition=0,
    seed=42,
    task_family=example.task_family,
    output_mode=example.output_mode,
    language=example.language,
    source_text=source_text,
    references=example.references,
    parent_table=example.parent_table,
    request=GENERIC_REQUEST,
    generated_text=raw_text,
    backend=GenerationBackend.CALLABLE,
    elapsed_seconds=raw_elapsed,
    input_tokens=getattr(usage, "prompt_tokens", None) if usage else None,
    output_tokens=getattr(usage, "completion_tokens", None) if usage else None,
    total_tokens=getattr(usage, "total_tokens", None) if usage else None,
    metadata={
        "generic_contract_test": True,
        "explicit_contract_supplied": False,
        "model": raw_model,
        "baseline_type": "raw_single_llm_generic",
    },
)

records.append(raw_record)
print(f"raw_generic done in {raw_elapsed:.1f}s")
show("raw_generic", raw_record.generated_text)

# =========================
# save + score
# =========================

generations_path = project_dir / f"evaluation/generations/{experiment_name}_generations.jsonl"
write_generation_records(generations_path, records)

metrics_payload = json.loads(paths["metric_config"].read_text(encoding="utf-8"))
metrics_payload["baseline_variant"] = "raw_generic"
metrics_payload["generations_path"] = str(generations_path)

metrics_path = project_dir / f"evaluation/config/metrics_{experiment_name}.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

scores_path = project_dir / f"evaluation/results/{experiment_name}_reference_metrics.jsonl"

print("Scoring reference metrics...")
scores = score_reference_metrics_for_notebook(
    project_dir,
    generations_path=generations_path,
    metric_config_path=metrics_path,
    output_path=scores_path,
    include_ineligible=True,
)

print("Saved generations:", generations_path)
print("Saved metrics:", scores_path)

display(pd.DataFrame([r.model_dump(mode="json") for r in records])[
    ["dataset_id", "example_id", "variant_id", "release_status", "writer_mode", "elapsed_seconds", "error"]
])

display(metric_table(scores))